# Apex Retail Intelligence — Gold Layer & KPI Reporting Notebook

**Author:** Aashi Phulera
**Programme:** Celebal Technologies | CEI'26 Internship Programme — Major Project
**Layer:** Gold (Star Schema) + Business Reporting
**Technology:** PySpark, Unity Catalog, Spark SQL

---

### Purpose
This notebook implements **Phase 5** and **Phase 6**:

**Phase 5 — Gold Star Schema:**
- `dim_customer` — customer attributes with SCD Type 2 history
- `dim_product` — product catalogue details
- `dim_promotion` — promotion types and identifiers
- `dim_date` — calendar dimension with year/month/day-of-week/week-of-year attributes
- `fact_sales` — central fact table joining transaction metrics to all dimension surrogate keys
- All tables registered in Unity Catalog under the `GOLD_tables` schema

**Phase 6 — KPI Reporting:**
1. Net Margin by Region
2. Average Order Value (AOV) by Promotion
3. Demographic Churn Heatmap (state × loyalty programme)
4. Product Quality Index (return rate by category)
5. Store Traffic by Hour and Day of Week

All KPIs are computed using native PySpark DataFrame/SQL operations and rendered entirely within this notebook — **no external BI tools used**, per assignment constraints.

### Prerequisites
Requires `03_Silver_Layer_Notebook` to have run first.

### Independently Executable
This notebook reads directly from Silver Delta tables (`apex_retail.silver.*`) and writes to `apex_retail.GOLD_tables.*` — no dependency on in-memory variables from other notebooks.

In [0]:
# ============================================
# PHASE 5: GOLD LAYER — STAR SCHEMA
# ============================================
from pyspark.sql.functions import explode, sequence, to_date, year, month, dayofweek, weekofyear

dim_customer = spark.table("apex_retail.silver.customers").select(
    "customer_sk", "customer_id", "age", "gender", "income_bracket", "loyalty_program",
    "customer_city", "customer_state", "is_active", "effective_start_date", "effective_end_date"
)
dim_customer.write.format("delta").mode("overwrite").saveAsTable("apex_retail.GOLD_tables.dim_customer")
print(f"✅ dim_customer: {dim_customer.count()} rows")

dim_product = spark.table("apex_retail.silver.products").select(
    "product_sk", "product_id", "product_name", "product_brand",
    "product_category", "unit_price", "product_return_rate"
)
dim_product.write.format("delta").mode("overwrite").saveAsTable("apex_retail.GOLD_tables.dim_product")
print(f"✅ dim_product: {dim_product.count()} rows")

dim_promotion = spark.table("apex_retail.silver.sales").select(
    "promotion_id", "promotion_type"
).dropDuplicates(["promotion_id"])
dim_promotion.write.format("delta").mode("overwrite").saveAsTable("apex_retail.GOLD_tables.dim_promotion")
print(f"✅ dim_promotion: {dim_promotion.count()} rows")

date_range = spark.sql("SELECT sequence(to_date('2018-01-01'), to_date('2027-12-31'), interval 1 day) as date")
dim_date = date_range.withColumn("date", explode("date")) \
    .withColumn("year", year("date")).withColumn("month", month("date")) \
    .withColumn("day_of_week", dayofweek("date")).withColumn("week_of_year", weekofyear("date"))
dim_date.write.format("delta").mode("overwrite").saveAsTable("apex_retail.GOLD_tables.dim_date")
print(f"✅ dim_date: {dim_date.count()} rows")

fact_sales = spark.table("apex_retail.silver.sales") \
    .join(dim_customer.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(dim_product.select("product_id", "product_sk"), "product_id", "left")
fact_sales.write.format("delta").mode("overwrite").saveAsTable("apex_retail.GOLD_tables.fact_sales")
print(f"✅ fact_sales: {fact_sales.count()} rows")

print("\n📋 Gold tables registered:")
display(spark.sql("SHOW TABLES IN apex_retail.GOLD_tables"))

✅ dim_customer: 1050 rows
✅ dim_product: 1041 rows
✅ dim_promotion: 864 rows
✅ dim_date: 3652 rows
✅ fact_sales: 2000 rows

📋 Gold tables registered:


database,tableName,isTemporary
gold_tables,dim_customer,false
gold_tables,dim_date,false
gold_tables,dim_product,false
gold_tables,dim_promotion,false
gold_tables,fact_sales,false


In [0]:
# ============================================
# PHASE 6: BUSINESS REPORTING — KPI GENERATION
# ============================================
from pyspark.sql.functions import sum as _sum, avg, count, desc, col

fact_sales = spark.table("apex_retail.GOLD_tables.fact_sales")
dim_customer = spark.table("apex_retail.GOLD_tables.dim_customer")
dim_product = spark.table("apex_retail.GOLD_tables.dim_product")

# KPI 1: Net Margin by Region
print("KPI 1: Net Margin by Region")
kpi1 = fact_sales.groupBy("store_location") \
    .agg((_sum("total_sales") - _sum("discount_applied")).alias("net_margin")) \
    .orderBy(desc("net_margin"))
display(kpi1)

KPI 1: Net Margin by Region


store_location,net_margin
Location D,1284153.76
Location B,1240979.8300000015
Location C,1183410.4199999995
Location A,1141099.970000001
Unknown,675482.7400000002


In [0]:
from pyspark.sql.functions import when
# KPI 2: Average Order Value (AOV) by Promotion
print("KPI 2: Average Order Value by Promotion")
kpi2 = fact_sales.groupBy("promotion_type") \
    .agg(avg("total_sales").alias("avg_order_value")) \
    .orderBy(desc("avg_order_value"))
display(kpi2)

# KPI 3: Demographic Churn Heatmap (state x loyalty program)
print("\nKPI 3: Demographic Churn Heatmap")
kpi3 = dim_customer.groupBy("customer_state", "loyalty_program") \
    .agg(
        count("*").alias("total_customers"),
        _sum(when(col("is_active") == False, 1).otherwise(0)).alias("churned_count")
    ) \
    .withColumn("churn_rate", col("churned_count") / col("total_customers")) \
    .orderBy(desc("churn_rate"))
display(kpi3)

# KPI 4: Product Quality Index (highest return rate categories)
print("\nKPI 4: Product Quality Index")
kpi4 = dim_product.groupBy("product_category") \
    .agg(avg("product_return_rate").alias("avg_return_rate")) \
    .orderBy(desc("avg_return_rate"))
display(kpi4)

# KPI 5: Store Traffic by Hour and Day
print("\nKPI 5: Store Traffic by Hour and Day of Week")
sales_silver = spark.table("apex_retail.silver.sales")
kpi5 = sales_silver.groupBy("transaction_hour", "day_of_week") \
    .agg(count("*").alias("transaction_count")) \
    .orderBy(desc("transaction_count"))
display(kpi5)

print("\n✅ Phase 6: All 5 KPIs computed successfully.")

KPI 2: Average Order Value by Promotion


promotion_type,avg_order_value
Flash Sale,3074.0150284629985
Buy One Get One Free,2857.8895978062164
20% Off,2795.723928571429
Unknown,2122.1987158469956



KPI 3: Demographic Churn Heatmap


customer_state,loyalty_program,total_customers,churned_count,churn_rate
State Y,No,166,0,0.0
State Z,Yes,161,0,0.0
State X,No,202,0,0.0
State X,Yes,161,0,0.0
State Z,No,180,0,0.0
State Y,Yes,180,0,0.0



KPI 4: Product Quality Index


product_category,avg_return_rate
Electronics,0.2674033149171271
Furniture,0.26688679245283026
Groceries,0.2560180995475113
Clothing,0.2393953488372091
Toys,0.23292452830188684



KPI 5: Store Traffic by Hour and Day of Week


transaction_hour,day_of_week,transaction_count
5,Wednesday,20
22,Thursday,19
5,Sunday,19
8,Thursday,18
7,Wednesday,18
22,Wednesday,17
20,Saturday,17
2,Sunday,17
1,Saturday,17
2,Saturday,17



✅ Phase 6: All 5 KPIs computed successfully.
